# FAST 动作分词器

## 动作归一化

In [51]:
# 每个动作维度的值（比如关节角度或速度）映射到[-1, 1]的范围，基于训练数据集的1%和99%分位数

In [52]:
import numpy as np
import torch

### ActionNormalizer

In [53]:
# raw_actions: [B, T, D]
class ActionNormalizer:
    def __init__(self, low_percent=1, high_percent=99):
        self.high_percent = high_percent
        self.low_percent = low_percent
        self.high = None
        self.low = None

    def normalize(self, raw_actions):
         assert len(raw_actions.shape) == 3
         raw_actions = raw_actions.cpu().numpy()

         B, T, D = raw_actions.shape
         raw_actions = raw_actions.reshape(-1, D) # 转换为 [num_actions, D]

         # 对每个动作维度单独求分位数并归一化
         self.low, self.high = np.percentile(raw_actions, [self.low_percent, self.high_percent], axis=0)
         norm_actions = (raw_actions - self.low) / (self.high - self.low)

         norm_actions = norm_actions.reshape(B, T, D)

         return torch.from_numpy(norm_actions).float()

    def denormalize(self, norm_actions):
        recon_actions = norm_actions * (self.high - self.low) + self.low
        return recon_actions

## 离散余弦变换 (DCT)

In [54]:
import math

### DCT

In [55]:
class DCT:
    def __init__(self, scale=10):
        """
        actions: [T, D]
        :param scale: 缩放因子
        """
        self.scale = scale

    # X[k] = sqrt(2/N) a_k Σ_{n:0->N-1} x[n] cos((πk/N)(n + 1/2))
    def dct_1d(self, x):
        """一维DCT"""
        N = len(x)
        X = []

        # 预计算缩放因子
        scale_0, scale_other = math.sqrt(1.0/N), math.sqrt(2.0/N)

        for k in range(N):
            scale = scale_0 if k == 0 else scale_other
            cos_sum = sum(x[n] * math.cos( math.pi * k *(n + 0.5) / N )
                          for n in range(N))
            X.append(cos_sum * scale)

        return torch.stack(X) # 标量 -> 向量

    def dct_per_dim(self, actions):
        """actions [T, D]"""
        T, D = actions.shape
        dct_freqs = torch.zeros_like(actions, dtype=torch.float64)

        for d in range(D):
            dct_freqs[:, d] = self.dct_1d(actions[:, d])

        return dct_freqs

    def __call__(self, x):
        dct_freqs = self.dct_per_dim(x)
        return torch.round(dct_freqs / self.scale).long() # scale and round

### IDCT

In [56]:
class IDCT(DCT):
    def __init__(self):
        super().__init__()

    # x[n] = sqrt(1/N) * X[0] + sqrt(2/N) * Σ_{k:1->N-1} X[k] cos((πk/N)*(n + 1/2))
    def idct_1d(self, X):
        N = len(X)
        x = []

        scale_0, scale_other = math.sqrt(1.0/N), math.sqrt(2.0/N)
        head = scale_0 * X[0]

        for n in range(N):
            cos_sum = scale_other * sum(X[k] * math.cos(math.pi * k * (n + 0.5) / N) for k in range(1, N))
            x.append(cos_sum + head)

        return torch.stack(x)

    def idct_per_dim(self, freqs):
        T, D = freqs.shape
        results = torch.zeros_like(freqs, dtype=torch.float64)

        for d in range(D):
            results[:, d] = self.idct_1d(freqs[:, d])

        return results

    def __call__(self, sparse_matrix):
        freqs = sparse_matrix * self.scale
        return self.idct_per_dim(freqs)

## 字节对编码 (BPE)

In [57]:
from collections import Counter
import einops

### BPE

In [58]:
class BPE:
    def __init__(self, vocab_size, min_freq_threshold):
        self.vocab_size = vocab_size
        self.min_freq_threshold = min_freq_threshold
        self.merge_rules = {}   # e.g. {(1, 2): 5}
        self.inverse_merge_rules = {}   # e.g. {5: {1, 2}}
        self.next_available_id = None

    def _flatten_by_frequency(self, x):
        """按频率顺序展平（低频系数优先）"""
        x = einops.rearrange(x, 'freq d -> (freq d)')
        return x.cpu().tolist()

    def fit(self, dct_freqs):
        """sequence: list of 2D tensor"""
        # 1. 按频率优先展平
        sequence = self._flatten_by_frequency(dct_freqs)

        # 2. 确定初始 ID 范围
        self.next_available_id = max(sequence) + 1

        # 3. 迭代合并，直到 词表足够小，或者频率都小于阈值
        cur_vocab_size = len(set(sequence))
        while cur_vocab_size < self.vocab_size:
            # 统计相邻 token 频率
            pair_freqs = Counter()
            for i in range(len(sequence) - 1):
                pair_freqs[(sequence[i], sequence[i + 1])] += 1

            # 最频繁的 pair
            most_common_pair, max_freq = pair_freqs.most_common(1)[0]
            if max_freq < self.min_freq_threshold:
                break

            # 分配新 ID
            new_id = self.next_available_id
            self.next_available_id += 1
            cur_vocab_size += 1

            # 记录合并规则
            self.merge_rules[most_common_pair] = new_id
            self.inverse_merge_rules[new_id] = most_common_pair

            # 应用合并
            new_sequence = []
            i = 0
            while i < len(sequence):
                # 如果匹配到
                if i < len(sequence) - 1 and (sequence[i], sequence[i + 1]) == most_common_pair:
                    new_sequence.append(new_id)
                    i += 2
                # 如果没有匹配到
                else:
                    new_sequence.append(sequence[i])
                    i += 1

            # 更新 sequence
            sequence = new_sequence

        return self

    def encode(self, dct_freqs):
        sequence = self._flatten_by_frequency(dct_freqs)

        # 从开头不断相邻匹配，直到无法匹配，从下一个开始相邻匹配
        i = 0
        while i < len(sequence) - 1:
            pair = (sequence[i], sequence[i + 1])
            if pair in self.merge_rules:
                new_id = self.merge_rules[pair]
                # 将 pair 替换为 new_id
                sequence[i: i+2] = [new_id]
            else:
                i += 1

        return torch.tensor(sequence, dtype=torch.long, device=dct_freqs.device)

    def decode(self, token_ids, T=None, D=None):
        assert T is not None or D is not None

        ids = token_ids.cpu().tolist()

        def flatten_nested_tuple(nested_tuple):
            if all(isinstance(x, int) for x in nested_tuple):
                return list(nested_tuple)

            result = []
            for item in nested_tuple:
                if isinstance(item, (tuple, list)):
                    flatten_item = flatten_nested_tuple(item)
                    result.extend(flatten_item)
                elif isinstance(item, int):
                    result.append(item)
                else:
                    raise TypeError(f'Tuple 内部必须为 int 或 tuple')

            return result

        nested_tuple = tuple(id if id not in self.inverse_merge_rules
                             else self.inverse_merge_rules[id]
                             for id in ids)

        results = flatten_nested_tuple(nested_tuple)

        # 恢复为 2D tensor
        results = torch.tensor(results, dtype=torch.long, device=token_ids.device)

        results = einops.rearrange(results, '(freq d) -> freq d', freq=T) if T is not None \
                    else einops.rearrange(results, '(freq d) -> freq d', d=D)

        return results

## 动作分词器

### FAST

In [59]:
class FAST:
    def __init__(self, dct, idct, bpe):
        self.dct = dct
        self.idct = idct
        self.bpe = bpe

    def fit(self, train_dataloader):
        for imgs, texts, actions in train_dataloader:
            # actions: [B, T, D] -> seq [T, D]
            for action_sequence in actions:
                dct_freqs = self.dct(action_sequence)
                self.bpe.fit(dct_freqs)

        return self

    def get_token_len(self, actions):
        """返回编码后的token长度"""
        sample = self.dct(actions[0])
        tokens = self.bpe.encode(sample)
        return len(tokens)

    def __call__(self, actions):
        # 如果已经是 token_ids (二维)，直接返回
        if len(actions.shape) == 2:
            return actions

        assert len(actions.shape) == 3

        batch_tokens = []
        for action_sequence in actions:
            dct_freqs = self.dct(action_sequence)
            token_ids = self.bpe.encode(dct_freqs)  # tensor
            batch_tokens.append(token_ids)

        # 填充token至相同长度
        from torch.nn.utils.rnn import pad_sequence
        padded_tokens = pad_sequence(batch_tokens, batch_first=True)

        return padded_tokens

    def decode(self, token_ids, T=None, D=None):
        assert len(token_ids.shape) == 2

        # bpe.decode: list of 1D -> list of 2D
        ibpe_tokens = [self.bpe.decode(tokens, T=T, D=D) for tokens in token_ids]

        # idct: -> list of 2D
        idct_tokens = [self.idct(tokens) for tokens in ibpe_tokens]

        return torch.stack(idct_tokens, dim=0)

## 自回归头

In [60]:
import torch.nn as nn

### AutoRegressionConfig


In [61]:
from dataclasses import dataclass

@dataclass
class AutoRegressionConfig:
    # 条件维度
    condition_dim: int = 128

    # d_model，动作向量的维度，同时也是 Transformer的接口维度
    d_model: int = 256

    # Transformer 的中间维度
    d_hidden: int = 512

    # 最大位置编码数
    max_pos_len: int = 1000

    # 模型组成
    n_layers: int = 3
    dropout_rate: float = 0.1
    n_heads: int = 8

    # 动作词表
    action_vocab_size: int = 10000

### MultiHeadAttention

In [62]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0

        self.d_model: int = config.d_model
        self.n_heads: int = config.n_heads
        self.d_k: int = config.d_model // config.n_heads

        self.W_Qs = nn.ModuleList([
            nn.Linear(self.d_model, self.d_k, bias=False) for _ in range(self.n_heads)
        ])

        self.W_Ks = nn.ModuleList([
            nn.Linear(self.d_model, self.d_k, bias=False) for _ in range(self.n_heads)
        ])

        self.W_Vs = nn.ModuleList([
            nn.Linear(self.d_model, self.d_k, bias=False) for _ in range(self.n_heads)
        ])

        self.W_O = nn.Linear(self.d_model, self.d_model, bias=False)

    def scaled_dot_attention(self, Q, K, V, mask=None):
        # Attention = Softmax((Q @ K.T) / sqrt(d_k)) @ V
        d_k = Q.shape[-1]

        scores = Q @ K.transpose(-1, -2) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        similarity = torch.softmax(scores, dim=-1)
        attention = similarity @ V

        return attention

    def forward(self, Q, K=None, V=None, mask=None):
        if K is None and V is None:
            K = V = Q

        return self.W_O(torch.cat([
            self.scaled_dot_attention(W_Q(Q), W_K(K), W_V(V), mask=mask)
            for W_Q, W_K, W_V in zip(self.W_Qs, self.W_Ks, self.W_Vs)
        ], dim=-1))

### DecoderBlock

In [63]:
class AdaptedDecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.masked_self_attention = MultiHeadAttention(config)

        self.feed_forward = nn.Sequential(
            nn.Linear(config.d_model, config.d_hidden),
            nn.GELU(),
            nn.Dropout(config.dropout_rate),
            nn.Linear(config.d_hidden, config.d_model),
        )

        self.layer_norm1 = nn.LayerNorm(config.d_model)
        self.layer_norm2 = nn.LayerNorm(config.d_model)

    def _get_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)
        return mask

    def forward(self, X, mask=None):
        # 标准数据 (B, seq_len, feature)
        if mask is None:
            mask = self._get_causal_mask(X.shape[1], X.device)

        # Pre-LayerNorm: Self_Attention(LayerNorm(X)) + X
        X = self.masked_self_attention(self.layer_norm1(X), mask=mask) + X
        X = self.feed_forward(self.layer_norm2(X)) + X

        return X

### AutoRegressiveHead

In [64]:
class AutoRegressiveHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # 条件编码, condition_dim 为条件维度, d_model 为编码维度，对应 action 编码维度
        self.condition_proj = nn.Linear(config.condition_dim, config.d_model)

        # action token 编码
        self.action_embedding = nn.Embedding(config.action_vocab_size, config.d_model)

        # 位置编码
        self.position_embedding = nn.Embedding(config.max_pos_len, config.d_model)

        # Transformer 解码
        self.decoder = nn.Sequential(
            *[AdaptedDecoderBlock(config) for _ in range(config.n_layers)],
        )

        # 输出层，将结果映射到词表
        self.fc_out = nn.Linear(config.d_model, config.action_vocab_size)

    def forward(self, action_tokens, conditions):
        """
        :param action_tokens: 动作序列token [B, T]，整数 token id
        :param conditions: VLM 输出的条件 [B, d_cond]
        :return: pred_actions 预测的动作序列token [B, T]
        """
        B, T = action_tokens.shape
        # 动作嵌入 + 条件嵌入
        x = self.action_embedding(action_tokens)
        cond = self.condition_proj(conditions).unsqueeze(1)

        # 拼接动作和条件，条件Prefix
        x = torch.cat([cond, x], dim=1)

        # 位置编码
        position_ids = torch.arange(T+1, device=action_tokens.device)
        pos = self.position_embedding(position_ids)
        pos = einops.repeat(pos, 't d -> b t d', b=B)
        x = x + pos

        # 解码
        x = self.decoder(x)

        # 输出结果，去除条件位置
        logits =  self.fc_out(x[:, 1:, :])
        return logits

## 整合到 VLA

### VLAWithAutoRegression

In [65]:
class VLAWithAutoRegression(nn.Module):
    def __init__(self, config, vlm, ar_head, action_tokenizer):
        super().__init__()
        self.config = config
        self.vlm_backbone = vlm
        self.ar_head = ar_head
        self.action_tokenizer = action_tokenizer

    def forward(self, imgs, texts, actions):
        """
        :param imgs: [B, C, H, W]
        :param texts: list of strings
        :param actions: [B, T, D]
        :return: logits: [B, T, vocab_size]
        """
        # CLIP 返回 [B, 512] 的文本特征
        conditions = self.vlm_backbone(imgs, texts)  # [B, 512]

        # 动作 tokenization
        action_tokens = self.action_tokenizer(actions)  # [B, L]

        # 自回归预测
        logits = self.ar_head(action_tokens, conditions)  # [B, L, vocab_size]

        return logits

### VLATrainerConfig

In [66]:
@dataclass
class VLATrainerConfig:
    device: torch.device = 'cuda' if torch.cuda.is_available() else 'cpu'

    num_epochs: int = 50

    # 采样温度
    temperature: float = 1.0

    lr: float = 1e-3

### VLAWithARHeadTrainer

In [67]:
class VLAWithARHeadTrainer:
    def __init__(self, config, model, optimizer, scheduler=None):
        self.config = config
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler

    def fit(self, train_dataloader):
        self.model.train()

        for epoch in range(self.config.num_epochs):
            running_loss = 0.0
            for i, (imgs, texts, actions) in enumerate(train_dataloader, 1):
                imgs, actions = imgs.to(self.config.device), actions.to(self.config.device)

                # 输入前 T - 1步，预测后 T - 1 步
                pred_logits = self.model(imgs, texts, actions[:, :-1]) # pred_logits [B, T-1, vocab_size]
                pred_logits = einops.rearrange(pred_logits, 'b t v -> (b t) v')

                targets = self.model.action_tokenizer(actions[:, 1:])

                # 转为 交叉熵损失 接受的形式
                targets = einops.rearrange(targets, 'b t -> (b t)')

                if hasattr(self.optimizer, 'train'):
                    self.optimizer.train()

                loss = torch.nn.functional.cross_entropy(pred_logits, targets)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                running_loss += loss.item()

                print(f'Epoch {epoch} | {self.config.num_epochs}, Step: {i} | {len(train_dataloader)} | Loss: {running_loss / i}')

            if self.scheduler is not None:
                self.scheduler.step()

    @torch.no_grad()
    def inference(self, imgs, texts, num_tokens):
        """
        预测未来 若干 个动作序列 token
        :return pred_action_tokens [B, max_pred_len]
        """
        self.model.eval()

        # 初始化空 action tokens
        B = imgs.shape[0]
        history = torch.zeros(B, 1, dtype=torch.long, device=self.config.device)

        # 自回归串行预测
        for _ in range(num_tokens):
            # 输入前 N 个历史动作，输出后 N 个动作，其中最后一个动作为预测
            logits = self.model(imgs, texts, history)
            next_logit = logits[:, -1, :] / self.config.temperature

            # 预测并采样
            probs = torch.nn.functional.softmax(next_logit, dim=-1)
            next_token = torch.multinomial(probs, 1) # [B, 1]

            history = torch.cat([history, next_token], dim=1)

        return history[:, 1:]

## 模拟数据测试

In [68]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from collections import Counter
import einops
from torch.nn.utils.rnn import pad_sequence
from transformers import CLIPProcessor, CLIPModel

In [69]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [70]:
#  1. CLIP模型
class CLIPVLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.output_dim = 512

    def forward(self, imgs, texts):
        inputs = self.processor(text=texts, images=imgs, return_tensors="pt", padding=True)
        inputs = {k: v.to(imgs.device) for k, v in inputs.items()}
        outputs = self.model(**inputs)
        return outputs.text_embeds

vlm = CLIPVLM().to(device)

In [75]:
class ComplexRobotDataset(Dataset):
    def __init__(self, num_samples=100, T=50, D=7):
        self.num_samples = num_samples
        self.T = T
        self.D = D
        self.instructions = [
            "pick up the red cup",
            "open the top drawer",
            "push the blue button",
            "grasp the yellow block",
            "rotate gripper clockwise",
            "move arm to left",
            "close the gripper softly"
        ]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 图像
        img = torch.rand(3, 224, 224)

        # 文本
        text = self.instructions[idx % len(self.instructions)]

        # 复杂动作：不同频率的正弦波叠加 + 噪声
        t = np.linspace(0, 4*np.pi, self.T)
        action = np.zeros((self.T, self.D))

        for d in range(self.D):
            # 基础运动
            base = 0.5 * np.sin(t * (0.5 + 0.2*d))
            # 中频抖动
            mid = 0.2 * np.sin(t * (3 + 0.5*d))
            # 高频噪声
            high = 0.05 * np.random.randn(self.T)
            # 偶尔的脉冲（模拟急停）
            if np.random.random() > 0.8:
                pulse_idx = np.random.randint(10, self.T-10)
                high[pulse_idx:pulse_idx+5] += 0.3

            action[:, d] = base + mid + high

        # 归一化到 [-1, 1] 左右
        action = action / np.max(np.abs(action)) * 0.9

        return img, text, torch.FloatTensor(action)

# 参数
dataset = ComplexRobotDataset(num_samples=1, T=50, D=7)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [76]:
# 自回归配置
ar_config = AutoRegressionConfig(
    condition_dim=512,
    d_model=64,
    action_vocab_size=20,
    n_layers=2,
    n_heads=4
)

# 训练配置
trainer_config = VLATrainerConfig(
    device=device,
    num_epochs=2,
    temperature=1.0,
    lr=1e-3
)

In [77]:
# 组件实例化
# ActionNormalizer
normalizer = ActionNormalizer(low_percent=1, high_percent=99)

# DCT/IDCT
dct = DCT(scale=10)
idct = IDCT()

# BPE
bpe = BPE(vocab_size=20, min_freq_threshold=2)

# FAST
fast = FAST(dct, idct, bpe)

# 自回归头
ar_head = AutoRegressiveHead(ar_config)

# VLA模型
vla = VLAWithAutoRegression(
    config=ar_config,
    vlm=vlm,
    ar_head=ar_head,
    action_tokenizer=fast
).to(device)


optimizer = torch.optim.Adam(vla.parameters(), lr=trainer_config.lr)
trainer = VLAWithARHeadTrainer(trainer_config, vla, optimizer)

print("所有组件实例化完成!")

所有组件实例化完成!


In [82]:
# 运行测试
# 训练FAST
print('开始训练 FAST')

fast.fit(dataloader)
print(f"BPE合并规则: {bpe.merge_rules}")
print('FAST 训练完成')

# 测试编码解码
imgs, texts, actions = next(iter(dataloader))
actions = actions.to(device)
tokens = fast(actions)
reconstructed = fast.decode(tokens, T=dataset.T, D=dataset.D)
print(f"重建MSE: {torch.nn.functional.mse_loss(actions, reconstructed):.6f}")

num_tokens = fast.get_token_len(actions)
print(f'BPE 编码得到 {num_tokens} tokens')

# 训练VLA
print('开始训练 VLA')
trainer.fit(dataloader)
print('VLA 训练完成')

print('开始推理')
# 推理
with torch.no_grad():
    test_imgs, test_texts, _ = next(iter(dataloader))
    test_imgs = test_imgs.to(device)
    tokens = trainer.inference(test_imgs, test_texts, num_tokens)
    actions = fast.decode(tokens, T=dataset.T, D=dataset.D)
    print(f"生成动作形状: {actions.shape}")

开始训练 FAST
BPE合并规则: {(0, 0): 1, (1, 1): 2, (2, 2): 3, (3, 3): 4, (4, 4): 5, (5, 5): 6, (6, 6): 7}
FAST 训练完成
ids: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
inverse_merge_rules: {1: (0, 0), 2: (1, 1), 3: (2, 2), 4: (3, 3), 5: (4, 4), 6: (5, 5), 7: (6, 6)}
nested_tuple: ((0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), 

## 跳转至结尾